# Prepare Environment
<hr>

In [ ]:
import os

# Remove the exported function definitions that cause the noise
for key in list(os.environ.keys()):
    if key.startswith("BASH_FUNC_"):
        del os.environ[key]

!rm .bedrock_agentcore.yaml

In [ ]:
!pip install -U -r requirements.txt --quiet

#### Create Demo Role

In [ ]:
import boto3
import json

ACCOUNT_ID = "869795389844"
ROLE_NAME = "AgentCoreDemoRole"

iam = boto3.client("iam")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "sagemaker.amazonaws.com",
                    "ec2.amazonaws.com",
                    "lambda.amazonaws.com",
                    "apigateway.amazonaws.com",
                    "bedrock-agentcore.amazonaws.com",
                ]
            },
            "Action": "sts:AssumeRole",
        },
    ],
}

permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "*",
            "Resource": "*",
        }
    ],
}

try:
    response = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Demo role for AgentCore notebook testing - wide open permissions",
    )
    role_arn = response["Role"]["Arn"]
    print(f"Created role: {role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    iam.update_assume_role_policy(
        RoleName=ROLE_NAME,
        PolicyDocument=json.dumps(trust_policy),
    )
    print(f"Role already exists, updated trust policy: {role_arn}")

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="DemoFullAccess",
    PolicyDocument=json.dumps(permissions_policy),
)

print(f"Attached permissions policy")
print(f"\nRole ARN: {role_arn}")

# Standalone Agent on Local Environment (Jupyter Notebook)
<hr>

In [ ]:
import json
import argparse

from strands import Agent
from strands.models import BedrockModel

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)

agent = Agent(
    model=model,
    system_prompt="You're a helpful assistant."
)

response = agent("Top 5 biggest countries by size ?")

In [ ]:
response = agent("What is the weather in Athens now ?")

#### Adding a tool

In [ ]:
from strands_tools.tavily import tavily_search

os.environ["TAVILY_API_KEY"] = "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"
#TODO : Add DUCKDUODO scenario

agent = Agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You're a helpful assistant."
)

In [ ]:
response = agent("What is the weather in Athens now ?")

#### Adding a custom tool

In [ ]:
from strands import tool

# Weather tool
@tool
def weather(city: str):
    """Get weather information.

    Args:
        city: City for which weather will be returned

    Returns:
        Weather of provided city as a string.
    """
    print(city)
    if city.lower() == "athens":
        weather = "very sunny"
    else:
        weather = "sunny"
    return weather

In [ ]:
agent = Agent(
    model=model,
    tools=[weather, tavily_search],
    system_prompt="You're a helpful assistant."
)

response = agent("What is the weather in Athens now ?")

In [ ]:
response = agent("What is the weather in Athens now ? Cross check with information from the web")

# Standalone Agent on AgentCore
<hr>

In [ ]:
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime


boto_session = Session()
region = boto_session.region_name

<img src="images/agent_core_configure_and_launch.png" width="1100">

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="01_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
invoke_response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check with information from the web."})
invoke_response

# MCP Server on  AgentCore
<hr>

In [ ]:
mcp_server_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = mcp_server_runtime.configure(
    entrypoint="01_mcp_server.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name='mpc_server',
)

In [ ]:
mcp_server_runtime.launch(auto_update_on_conflict=True)

# Agent -> MCP Server on  AgentCore
<hr>

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="02_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check your answer with web information."})
response

# MCP Gateway
<hr>

#### Create Gateway

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control")

# Create new gateway
response = agentcore_control_client.create_gateway(
    name='gateway',
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
)

gateway_id = response["gatewayId"]
gateway_url = response["gatewayUrl"]

print(f"Gateway URL: {gateway_url}")

#### Create gateway MCP target

In [ ]:
MCP_SERVER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:runtime/mpc_server-IbPAU23Iwf"
encoded_arn = MCP_SERVER_ARN.replace(":", "%3A").replace("/", "%2F")

mcp_endpoint = (
    f"https://bedrock-agentcore.{region}.amazonaws.com"
    f"/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

target = agentcore_control_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name='mcp-server',
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_endpoint,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "GATEWAY_IAM_ROLE",
            "credentialProvider": {
                "iamCredentialProvider": {
                    "service": "bedrock-agentcore",
                    "region": region,
                }
            },
        }
    ],
)

print(f"Target ID: {target['targetId']}")
print(f"Target status: {target['status']}")

#### Create gateway REST API target

In [ ]:
ACCOUNT_ID = "869795389844"
S3_OPENAPI_URI = f"s3://agentcore-rest-api-{ACCOUNT_ID}/openapi.json"
REST_API_KEY_PROVIDER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:token-vault/default/apikeycredentialprovider/rest-api-key"

print(f"\nAdding REST API target...")
rest_target = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="rest-api",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "s3": {
                    "uri": S3_OPENAPI_URI,
                    "bucketOwnerAccountId": ACCOUNT_ID,
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": REST_API_KEY_PROVIDER_ARN,
                    "credentialLocation": "HEADER",
                    "credentialParameterName": "x-api-key",
                }
            },
        }
    ],
)

print(f"REST API Target ID: {rest_target['targetId']}")
print(f"REST API Target status: {rest_target['status']}")

# Agent -> MCP Gateway -> MCP on AgentCore
<hr>

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="03_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check with web info."})
response

# Clean Up